<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model

- Given the high prevalence of missing values and the likely non-linear relationships between features and the target, a boosting-based approach is well suited for this problem. I chose XGBoost due to its strong track record in handling imbalanced classification tasks and its flexibility in capturing complex, non-linear patterns.

- A common alternative in credit risk modeling is Logistic Regression; however, in its standard form it assumes linear decision boundaries. Given the observed feature behavior, this assumption is likely too restrictive for this dataset (though not formally tested here).

- The model is optimized using `log_loss`, while hyperparameter tuning is guided by `PR-AUC`, which is more appropriate for highly imbalanced datasets.

- This approach provides several advantages:
  - ability to model non-linear relationships  
  - native handling of missing values without requiring imputation  
  - robustness to multicollinearity among features  

- Model performance on the test set is moderate. In the context of risk scoring, correctly identifying positive cases (defaults) is typically more critical than minimizing false positives. Therefore, Recall is considered more important than Precision in this setting.



- Categorical features with NaNs
    - account_status,
    - account_worst_status_0_3m,
    - account_worst_status_12_24m,
    - account_worst_status_3_6m,
    - account_worst_status_6_12m,
    - num_arch_written_off_0_12m,
    - num_arch_written_off_12_24m,
    - worst_status_active_inv

- Categorical features no NaNs
    - merchant_category,
    - merchant_group,
    - has_paid,
    - name_in_email,
    - num_arch_dc_0_12m,
    - num_arch_dc_12_24m,
    - status_last_archived_0_24m, 
    - status_2nd_last_archived_0_24m,
    - status_3rd_last_archived_0_24m,
    - status_max_archived_0_6_months',
    - status_max_archived_0_12_months,
    - status_max_archived_0_24_months

- Numerical features with NaNs
    - account_days_in_dc_12_24m,
    - account_days_in_rem_12_24m,
    - account_days_in_term_12_24m,
    - account_incoming_debt_vs_paid_0_24m,
    - avg_payment_span_0_12m,
    - avg_payment_span_0_3m,
    - num_active_div_by_paid_inv_0_12m

- Numerical features no NaNs
    - account_amount_added_12_24m,
    - age,
    - max_paid_inv_0_12m,
    - max_paid_inv_0_24m,
    - num_active_inv,
    - num_arch_ok_0_12m,
    - num_arch_ok_12_24m,
    - num_arch_rem_0_12m,
    - num_unpaid_bills,
    - recovery_debt,
    - sum_capital_paid_account_0_12m,
    - sum_capital_paid_account_12_24m,
    - sum_paid_inv_0_12m,
    - time_hours

</div>

#### Load Data

In [1]:


import joblib
import shap 
import pandas as pd
import numpy as np
import warnings
import category_encoders as ce
from statsmodels.stats.proportion import proportions_ztest
from sklearn.metrics import roc_auc_score, recall_score, precision_score, average_precision_score, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from preprocessing import QuantileBinner
from utils import evaluate_and_report

import category_encoders as ce
from xgboost import XGBClassifier

import utils
import importlib
importlib.reload(utils)
from utils import evaluate_and_report


from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import category_encoders as ce



warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)
pd.options.display.float_format = '{:.5f}'.format



df = pd.read_csv("./dataset.csv", sep = ";")
df['has_paid'] = df['has_paid'].astype(int)

df_nna = df[df['default'].notna()]  # observations for which we have default values (Training + Validation set)
#df_na = df[df['default'].isna()]    # observations for which we dont have default values (Unlabeled data)

X = df_nna.drop(columns=["default"])
y = df_nna["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)


/Users/aleksamihajlovic/Documents/pd-prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
len(df_nna)

89976

<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model 1: Logistic regression
- Numerical fetures
    - account_days_in_dc_12_24m - only 11% NAs, 0 is the median - impute median
    - account_days_in_rem_12_24m - only 11% NAs, 0 is the median - impute median
    - account_days_in_term_12_24m - only 11% NAs, 0 is the median - impute median
    - account_incoming_debt_vs_paid_0_24m - assign it median or -999
    - avg_payment_span_0_3m - assign it median or -999
    - avg_payment_span_0_12m - assign it median or -999
    - num_active_div_by_paid_inv_0_12m - assign it median or -999
    
- Categorical
    - account_satus - Categorical, make NA a new category -> target encoding
    - account_worst_status_0_3m - Categorical, make new category -> target encoding
    - account_worst_status_12_24m - Categorical, make new category -> target encoding
    - account_worst_status_3_6m - Categorical, make new category -> target encoding
    - account_worst_status_6_12m - Categorical, make new category -> target encoding
    - num_arch_written_off_0_12m - Categorical, make new category -> target encoding
    - num_arch_written_off_12_24m - Categorical, make new category -> target encoding
    - worst_status_active_inv - Categorical, make new category -> target encoding


</div>

In [3]:



# -------------------------
# FEATURE GROUPS
# -------------------------

cat_na_columns = [
    'account_status', 
    'account_worst_status_0_3m', 
    'account_worst_status_12_24m',
    'account_worst_status_3_6m', 
    'account_worst_status_6_12m',
    'num_arch_written_off_0_12m', 
    'num_arch_written_off_12_24m',
    'worst_status_active_inv'
]

cat_no_na_columns = [
    'merchant_category', 
    'merchant_group', 
    'has_paid', 
    'name_in_email',
    'num_arch_dc_0_12m', 
    'num_arch_dc_12_24m',
    'status_last_archived_0_24m', 
    'status_2nd_last_archived_0_24m',
    'status_3rd_last_archived_0_24m',
    'status_max_archived_0_6_months', 
    'status_max_archived_0_12_months',
    'status_max_archived_0_24_months'
]

num_na_columns = [
    'account_days_in_dc_12_24m', 
    'account_days_in_rem_12_24m',
    'account_days_in_term_12_24m', 
    'account_incoming_debt_vs_paid_0_24m',
    'avg_payment_span_0_12m', 
    'avg_payment_span_0_3m',
    'num_active_div_by_paid_inv_0_12m'
]

num_no_na_columns = [
    'account_amount_added_12_24m', 
    'age', 
    'max_paid_inv_0_12m',
    'max_paid_inv_0_24m', 
    'num_active_inv', 
    'num_arch_ok_0_12m',
    'num_arch_ok_12_24m', 
    'num_arch_rem_0_12m', 
    'num_unpaid_bills',
    'recovery_debt', 
    'sum_capital_paid_account_0_12m',
    'sum_capital_paid_account_12_24m', 
    'sum_paid_inv_0_12m',
    'time_hours'
]


all_cat_cols = cat_na_columns + cat_no_na_columns
all_num_cols = num_na_columns + num_no_na_columns
feature_cols = all_cat_cols + all_num_cols

# =========================
# KEEP ONLY MODEL FEATURES
# =========================

X_train_lr1 = X_train[feature_cols].copy()
X_test_lr1 = X_test[feature_cols].copy()

# -------------------------
# CREATE MISSING INDICATORS (IMPORTANT STEP)
# -------------------------

def add_missing_indicators(df, num_na_cols):
    df = df.copy()
    for col in num_na_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)
    return df

X_train_lr1 = add_missing_indicators(X_train_lr1, num_na_columns)
X_test_lr1 = add_missing_indicators(X_test_lr1, num_na_columns)

# indicator columns
num_indicator_columns = [f"{col}_missing" for col in num_na_columns]

# -------------------------
# PIPELINES
# -------------------------

# numeric with NA → impute + scale
num_na_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# numeric without NA → scale
num_no_na_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# indicators → passthrough (NO scaling!)
indicator_pipeline = Pipeline([
    ("passthrough", "passthrough")
])

# categorical → impute + WoE
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("woe", ce.WOEEncoder(handle_missing='value', handle_unknown='value'))
])

# -------------------------
# PREPROCESSOR
# -------------------------

preprocessor = ColumnTransformer([
    ("num_na", num_na_pipeline, num_na_columns),
    ("num_no_na", num_no_na_pipeline, num_no_na_columns),
    ("indicators", indicator_pipeline, num_indicator_columns),
    ("cat_woe", cat_pipeline, all_cat_cols),
])

# -------------------------
# MODEL
# -------------------------

model = Pipeline([
    ("preprocessing", preprocessor),
    ("clf", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    ))
])

# -------------------------
# HYPERPARAMETERS
# -------------------------

param_dist = {
    "clf__C": np.logspace(-3, 2, 10),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_lr1 = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="average_precision",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# -------------------------
# TRAIN
# -------------------------

random_search_lr1.fit(X_train_lr1, y_train)

# -------------------------
# EVALUATE
# -------------------------

evaluate_and_report(random_search_lr1, X_test_lr1, y_test, threshold=0.5, )

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END .......................................clf__C=0.001; total time=   1.6s
[CV] END .......................................clf__C=0.001; total time=   1.6s
[CV] END .......................................clf__C=0.001; total time=   1.6s
[CV] END .......................................clf__C=0.001; total time=   1.7s
[CV] END ........................clf__C=0.003593813663804626; total time=   1.7s
[CV] END .......................................clf__C=0.001; total time=   1.7s
[CV] END ........................clf__C=0.003593813663804626; total time=   1.8s
[CV] END ........................clf__C=0.003593813663804626; total time=   1.8s
[CV] END ........................clf__C=0.003593813663804626; total time=   1.9s
[CV] END ........................clf__C=0.003593813663804626; total time=   1.7s
[CV] END .........................clf__C=0.01291549665014884; total time=   1.3s
[CV] END .........................clf__C=0.01291

In [4]:
evaluate_and_report(random_search_lr1, X_test_lr1, y_test, threshold=0.8, metric="average_precision")

Best CV average_precision: 0.16126337413411362

Best Parameters:
clf__C: 100.0

Test ROC AUC: 0.9041476670328931
Test PR AUC: 0.15228472679021723
Precision: 0.14166666666666666
Recall: 0.5271317829457365

Confusion Matrix:
          Pred 0  Pred 1
Actual 0   16914     824
Actual 1     122     136


<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model 2: Log Regression
- WoE binning for NUMERICAL features
- Add ratios
- Log transforms
- Caps - winsorization
- try L1
- interaction features
- change class weight -> class_weight={0:1, 1:2}



</div>

In [ ]:
X["debt_to_paid"] = X["recovery_debt"] / (X["sum_paid_inv_0_12m"] + 1)
X["active_to_paid"] = X["num_active_inv"] / (X["sum_paid_inv_0_12m"] + 1)

X["log_recovery_debt"] = np.log1p(X["recovery_debt"])

X["recovery_debt"] = X["recovery_debt"].clip(upper=X["recovery_debt"].quantile(0.99))

X["debt_x_active"] = X["recovery_debt"] * X["num_active_inv"]

<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model 3: XGBoost
- No feature manipulation

</div>

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

# =========================
# FEATURE GROUPS
# =========================

cat_na_cols = [
    "account_status",
    "account_worst_status_0_3m",
    "account_worst_status_12_24m",
    "account_worst_status_3_6m",
    "account_worst_status_6_12m",
    "num_arch_written_off_0_12m",
    "num_arch_written_off_12_24m",
    "worst_status_active_inv",
]

cat_no_na_cols = [
    "merchant_category",
    "merchant_group",
    "has_paid",
    "name_in_email",
    "num_arch_dc_0_12m",
    "num_arch_dc_12_24m",
    "status_last_archived_0_24m",
    "status_2nd_last_archived_0_24m",
    "status_3rd_last_archived_0_24m",
    "status_max_archived_0_6_months",
    "status_max_archived_0_12_months",
    "status_max_archived_0_24_months",
]

num_na_cols = [
    "account_days_in_dc_12_24m",
    "account_days_in_rem_12_24m",
    "account_days_in_term_12_24m",
    "account_incoming_debt_vs_paid_0_24m",
    "avg_payment_span_0_12m",
    "avg_payment_span_0_3m",
    "num_active_div_by_paid_inv_0_12m",
]

num_no_na_cols = [
    "account_amount_added_12_24m",
    "age",
    "max_paid_inv_0_12m",
    "max_paid_inv_0_24m",
    "num_active_inv",
    "num_arch_ok_0_12m",
    "num_arch_ok_12_24m",
    "num_arch_rem_0_12m",
    "num_unpaid_bills",
    "recovery_debt",
    "sum_capital_paid_account_0_12m",
    "sum_capital_paid_account_12_24m",
    "sum_paid_inv_0_12m",
    "time_hours",
]

all_cat_cols = cat_na_cols + cat_no_na_cols
all_num_cols = num_na_cols + num_no_na_cols
feature_cols = all_cat_cols + all_num_cols

# =========================
# KEEP ONLY MODEL FEATURES
# =========================

X_train_xgb1 = X_train[feature_cols].copy()
X_test_xgb1 = X_test[feature_cols].copy()

# =========================
# CAST CATEGORICAL COLUMNS
# =========================

for col in all_cat_cols:
    X_train_xgb1[col] = X_train_xgb1[col].astype("category")
    X_test_xgb1[col] = X_test_xgb1[col].astype("category")
    X_test_xgb1[col] = X_test_xgb1[col].cat.set_categories(
        X_train_xgb1[col].cat.categories
    )

# =========================
# CLASS IMBALANCE
# =========================

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# =========================
# MODEL
# =========================

model = XGBClassifier(
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    enable_categorical=True,
    tree_method="hist",
)

# =========================
# HYPERPARAMETERS
# =========================

param_dist = {
    "n_estimators": [200, 300, 500, 1000],
    "max_depth": [3, 5, 8],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": np.linspace(0, 2, 5),
    "min_child_weight": [1, 3, 5, 10],
    "reg_lambda": np.linspace(0, 3, 5),
    "reg_alpha": np.linspace(0, 3, 5),
}

# =========================
# CROSS-VALIDATION
# =========================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_xgb1 = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    scoring="average_precision",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    error_score="raise",
)

# =========================
# TRAIN
# =========================

random_search_xgb1.fit(X_train_xgb1, y_train)

# =========================
# EVALUATE
# =========================

evaluate_and_report(
    random_search_xgb1,
    X_test_xgb1,
    y_test,
    threshold=0.5,
    metric="average_precision",
)

In [ ]:
evaluate_and_report( random_search_xgb1, X_test_xgb1, y_test, threshold=0.30, metric="average_precision")

<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model 4: XGBoost
- WoE Encoding for categorical and numerical values with NaNs 

</div>

In [ ]:


cat_na_cols = [
    "account_status",
    "account_worst_status_0_3m",
    "account_worst_status_12_24m",
    "account_worst_status_3_6m",
    "account_worst_status_6_12m",
    "num_arch_written_off_0_12m",
    "num_arch_written_off_12_24m",
    "worst_status_active_inv",
]

cat_no_na_cols = [
    "merchant_category",
    "merchant_group",
    "has_paid",
    "name_in_email",
    "num_arch_dc_0_12m",
    "num_arch_dc_12_24m",
    "status_last_archived_0_24m",
    "status_2nd_last_archived_0_24m",
    "status_3rd_last_archived_0_24m",
    "status_max_archived_0_6_months",
    "status_max_archived_0_12_months",
    "status_max_archived_0_24_months",
]

num_na_cols = [
    "account_days_in_dc_12_24m",
    "account_days_in_rem_12_24m",
    "account_days_in_term_12_24m",
    "account_incoming_debt_vs_paid_0_24m",
    "avg_payment_span_0_12m",
    "avg_payment_span_0_3m",
    "num_active_div_by_paid_inv_0_12m",
]

num_no_na_cols = [
    "account_amount_added_12_24m",
    "age",
    "max_paid_inv_0_12m",
    "max_paid_inv_0_24m",
    "num_active_inv",
    "num_arch_ok_0_12m",
    "num_arch_ok_12_24m",
    "num_arch_rem_0_12m",
    "num_unpaid_bills",
    "recovery_debt",
    "sum_capital_paid_account_0_12m",
    "sum_capital_paid_account_12_24m",
    "sum_paid_inv_0_12m",
    "time_hours",
]

all_cat_cols = cat_na_cols + cat_no_na_cols
all_num_cols = num_na_cols + num_no_na_cols
feature_cols = all_cat_cols + all_num_cols

# =========================
# KEEP ONLY MODEL FEATURES
# =========================

X_train_xgb2 = X_train[feature_cols].copy()
X_test_xgb2 = X_test[feature_cols].copy()

# IDENTIFY COLUMNS

na_columns = X_train_xgb2.columns[X_train_xgb2.isna().any()].tolist()

numeric_columns = X_train_xgb2.select_dtypes(include=np.number).columns.tolist()
cat_columns = [col for col in X_train_xgb2.columns if col not in numeric_columns]

num_na_columns = [col for col in na_columns if col in numeric_columns]
num_no_na_columns = [col for col in numeric_columns if col not in num_na_columns]


# PIPELINES

# numeric with NA → bin + WoE
num_na_pipeline = Pipeline([
    ("binning", QuantileBinner(n_bins=10)),
    ("woe", ce.WOEEncoder(handle_unknown='value', handle_missing='value'))
])

# numeric without NA → passthrough
num_pipeline = Pipeline([
    ("passthrough", "passthrough")
])

# categorical → WoE (NA handled internally)
cat_pipeline = Pipeline([
    ("encoder", ce.WOEEncoder(handle_unknown='value', handle_missing='value'))
])

preprocessor = ColumnTransformer([
    ("num_na_woe", num_na_pipeline, num_na_columns),
    ("num", num_pipeline, num_no_na_columns),
    ("cat_woe", cat_pipeline, cat_columns),
])

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = Pipeline([
    ("preprocessing", preprocessor),
    ("clf", XGBClassifier(
        #eval_metric="auc",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        use_label_encoder=False
    ))
])

param_dist = {
    "clf__n_estimators": [200,300,500,1000],
    "clf__max_depth": [3, 5, 8],
    "clf__learning_rate": [0.01, 0.02, 0.05, 0.1],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "clf__gamma": np.linspace(0, 2, 5),
    "clf__min_child_weight": [1, 3, 5, 10],
    "clf__reg_lambda": np.linspace(0, 3, 5),
    "clf__reg_alpha": np.linspace(0, 3, 5),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search_xgb2 = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    #scoring="roc_auc",
    scoring="average_precision",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# MODEL TRAINING
random_search_xgb2.fit(X_train, y_train)

evaluate_and_report(random_search_xgb2, X_test_xgb2, y_test, threshold=0.5, metric="average_precision")

In [ ]:
evaluate_and_report(random_search_xgb2, X_test_xgb2, y_test, threshold=0.4, metric="average_precision")

<div style="background-color:#fff8dc; padding:15px;  color:black;">

## Model 5: Deep Neural Network
- Median inpute for NaNs of numerical features
- Numerical features are standardized
- Categorical features are encoded

</div>

In [28]:

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset, DataLoader

# =========================
# 1. COPY DATA
# =========================

X_train_nn1 = X_train.copy()
X_test_nn1 = X_test.copy()

# =========================
# 2. DEFINE COLUMNS
# =========================

cat_columns = cat_na_columns + cat_no_na_columns
num_columns = num_na_columns + num_no_na_columns

# =========================
# 3. ENCODE CATEGORICALS (for embeddings)
# =========================


cat_encoders = {}

for col in cat_columns:
    le = LabelEncoder()
    
    # convert to string
    X_train_nn1[col] = X_train_nn1[col].astype(str)
    X_test_nn1[col] = X_test_nn1[col].astype(str)
    
    # fill NaN
    X_train_nn1[col] = X_train_nn1[col].fillna("MISSING")
    X_test_nn1[col] = X_test_nn1[col].fillna("MISSING")
    
    # 🔥 IMPORTANT: ensure "MISSING" is in training classes
    if "MISSING" not in X_train_nn1[col].unique():
        X_train_nn1.loc[X_train_nn1.index[0], col] = "MISSING"
    
    # fit encoder
    le.fit(X_train_nn1[col])
    
    # transform train
    X_train_nn1[col] = le.transform(X_train_nn1[col])
    
    # map unknown test values safely
    known_classes = set(le.classes_)
    X_test_nn1[col] = X_test_nn1[col].apply(
        lambda x: x if x in known_classes else "MISSING"
    )
    
    # transform test
    X_test_nn1[col] = le.transform(X_test_nn1[col])
    
    cat_encoders[col] = le

# =========================
# 4. NUMERICAL PREPROCESSING
# =========================



imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_nn1[num_columns] = imputer.fit_transform(X_train_nn1[num_columns])
X_test_nn1[num_columns] = imputer.transform(X_test_nn1[num_columns])

X_train_nn1[num_columns] = scaler.fit_transform(X_train_nn1[num_columns])
X_test_nn1[num_columns] = scaler.transform(X_test_nn1[num_columns])

# =========================
# 5. PYTORCH DATASET
# =========================



class TabularDataset(Dataset):
    def __init__(self, X, y, cat_cols, num_cols):
        self.X_cat = torch.tensor(X[cat_cols].values, dtype=torch.long)
        self.X_num = torch.tensor(X[num_cols].values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

# =========================
# 6. CREATE DATALOADERS
# =========================

train_dataset = TabularDataset(X_train_nn1, y_train, cat_columns, num_columns)
test_dataset = TabularDataset(X_test_nn1, y_test, cat_columns, num_columns)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# =========================
# 7. MODEL (with embeddings)
# =========================

import torch.nn as nn

class TabularNN(nn.Module):
    def __init__(self, cat_dims, num_dim):
        super().__init__()
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_dim, min(50, (cat_dim + 1)//2))
            for cat_dim in cat_dims
        ])
        
        emb_dim = sum([emb.embedding_dim for emb in self.embeddings])
        
        self.model = nn.Sequential(
            nn.Linear(emb_dim + num_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x_cat, x_num):
        emb = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embeddings)]
        x = torch.cat(emb + [x_num], dim=1)
        return self.model(x).squeeze(1)

# =========================
# 8. INITIALIZE MODEL
# =========================

cat_dims = [X_train_nn1[col].nunique() for col in cat_columns]
num_dim = len(num_columns)

model = TabularNN(cat_dims, num_dim)

# =========================
# 9. TRAINING SETUP
# =========================

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)

pos_weight = torch.tensor(
    [(y_train == 0).sum() / (y_train == 1).sum()],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# =========================
# 10. TRAINING LOOP
# =========================

EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for x_cat, x_num, y_batch in train_loader:
        optimizer.zero_grad()
        
        logits = model(x_cat, x_num)
        loss = criterion(logits, y_batch)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

# =========================
# 11. EVALUATION
# =========================




# =========================
# EVALUATION
# =========================

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for x_cat, x_num, y_batch in test_loader:
        logits = model(x_cat, x_num)
        probs = torch.sigmoid(logits)
        
        all_preds.extend(probs.numpy())
        all_targets.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# --- AUC metrics ---
roc = roc_auc_score(all_targets, all_preds)
pr = average_precision_score(all_targets, all_preds)

# --- threshold (default 0.5 for now) ---



Epoch 1, Loss: 1.0041
Epoch 2, Loss: 0.8532
Epoch 3, Loss: 0.7995
Epoch 4, Loss: 0.7596
Epoch 5, Loss: 0.7466
Epoch 6, Loss: 0.7317
Epoch 7, Loss: 0.7046
Epoch 8, Loss: 0.7064
Epoch 9, Loss: 0.6837
Epoch 10, Loss: 0.6957
Epoch 11, Loss: 0.6719
Epoch 12, Loss: 0.6580
Epoch 13, Loss: 0.6499
Epoch 14, Loss: 0.6439
Epoch 15, Loss: 0.6247
Epoch 16, Loss: 0.6291
Epoch 17, Loss: 0.6145
Epoch 18, Loss: 0.6060
Epoch 19, Loss: 0.6056
Epoch 20, Loss: 0.5929
Epoch 21, Loss: 0.5847
Epoch 22, Loss: 0.5851
Epoch 23, Loss: 0.5847
Epoch 24, Loss: 0.5690
Epoch 25, Loss: 0.5764
Epoch 26, Loss: 0.5623
Epoch 27, Loss: 0.5619
Epoch 28, Loss: 0.5469
Epoch 29, Loss: 0.5469
Epoch 30, Loss: 0.5348
Epoch 31, Loss: 0.5329
Epoch 32, Loss: 0.5189
Epoch 33, Loss: 0.5356
Epoch 34, Loss: 0.5413
Epoch 35, Loss: 0.5192
Epoch 36, Loss: 0.5160
Epoch 37, Loss: 0.5153
Epoch 38, Loss: 0.5325
Epoch 39, Loss: 0.5098
Epoch 40, Loss: 0.5005
Epoch 41, Loss: 0.5120
Epoch 42, Loss: 0.4934
Epoch 43, Loss: 0.4927
Epoch 44, Loss: 0.48

In [32]:
threshold = 0.60
y_pred = (all_preds >= threshold).astype(int)

# --- classification metrics ---
precision = precision_score(all_targets, y_pred)
recall = recall_score(all_targets, y_pred)

# --- confusion matrix ---
cm = confusion_matrix(all_targets, y_pred)

# =========================
# PRINT RESULTS
# =========================

print(f"Test ROC AUC: {roc:.4f}")
print(f"Test PR AUC: {pr:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

print("\nConfusion Matrix:")
print("          Pred 0  Pred 1")
print(f"Actual 0   {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"Actual 1   {cm[1,0]:5d}   {cm[1,1]:5d}")

Test ROC AUC: 0.8500
Test PR AUC: 0.1488
Precision: 0.1082
Recall: 0.4845

Confusion Matrix:
          Pred 0  Pred 1
Actual 0   16708    1030
Actual 1     133     125


In [ ]:
y_proba = random_search_lr1.best_estimator_.predict_proba(X_test)[:, 1]

from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1)

best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])

# Model Shipping

#### Serialize the model

In [ ]:
best_model = random_search_xgb1.best_estimator_
joblib.dump({
    "model": best_model,
    "features": X_train.columns.tolist()
}, "model_bundle.pkl")

#### Load and run the model

In [ ]:

bundle = joblib.load('./model_bundle.pkl')

mod = bundle["model"]
features = bundle["features"]

temp = pd.read_json("loan5.json")
temp = temp.reindex(columns=features)
temp["pd"] = mod.predict_proba(temp)[:, 1]

print(temp[["uuid", "pd"]])

#### Feature Importance with Shap

In [ ]:
xgb_model = mod.named_steps["clf"]
X_transformed = mod.named_steps["preprocessing"].transform(temp)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_transformed)
shap.summary_plot(
    shap_values,
    X_transformed,
    feature_names=features,
    plot_type="bar"
)